In [1]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("../data/raw/electronics_inventory.csv")

print("Original shape:", df.shape)

df.head()

Original shape: (16815, 14)


,Date,Product_ID,Product_Name,Category,Brand,Current_Stock,Units_Sold,Unit_Price,Discount_Percent,Promotion,Holiday,Lead_Time_Days,Safety_Stock,Stockout
0,2022-02-17,P001,iPhone 13,Mobile,Apple,154.0,9,65000.0,0.0,0.0,0,7.0,15,0
1,2026-06-06,P008,Samsung 55-inch Smart TV,TV,Samsung,101.0,3,120000.0,0.0,1.0,0,14.0,5,0
2,2025-06-04,P006,HP Pavilion 15,Laptop,HP,101.0,5,95000.0,0.0,1.0,0,9.0,8,0
3,2022-11-14,P004,Redmi Note 13,Mobile,Xiaomi,91.0,10,32000.0,0.0,0.0,0,5.0,20,0
4,2025-08-08,P001,iPhone 13,Mobile,Apple,129.0,5,65000.0,0.0,0.0,0,7.0,15,0


In [3]:
clean_df = df.copy()

print("Working dataset shape:", clean_df.shape)

Working dataset shape: (16815, 14)


In [4]:
clean_df["Date"] = pd.to_datetime(
    clean_df["Date"],
    errors="coerce"
)

print(clean_df.dtypes)

Date                datetime64[ns]
Product_ID                  object
Product_Name                object
Category                    object
Brand                       object
Current_Stock              float64
Units_Sold                   int64
Unit_Price                 float64
Discount_Percent           float64
Promotion                  float64
Holiday                      int64
Lead_Time_Days             float64
Safety_Stock                 int64
Stockout                     int64
dtype: object


In [5]:
clean_df["Date"].isna().sum()

np.int64(0)

In [6]:
duplicate_count = clean_df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 75


In [7]:
clean_df = clean_df.drop_duplicates().reset_index(drop=True)

print("Shape after removing duplicates:", clean_df.shape)

Shape after removing duplicates: (16740, 14)


In [8]:
print(clean_df.isnull().sum())

Date                  0
Product_ID            0
Product_Name          0
Category              0
Brand                 0
Current_Stock        66
Units_Sold            0
Unit_Price           66
Discount_Percent    133
Promotion            50
Holiday               0
Lead_Time_Days       50
Safety_Stock          0
Stockout              0
dtype: int64


Handle Unit_Price
- use the product's median historical price.

In [9]:
clean_df["Unit_Price"] = (
    clean_df.groupby("Product_ID")["Unit_Price"]
    .transform(lambda x: x.fillna(x.median()))
)

In [10]:
print("Missing Unit_Price:", clean_df["Unit_Price"].isnull().sum())

Missing Unit_Price: 0


Handle Discount_Percent


In [11]:
pd.crosstab(
    clean_df["Discount_Percent"].isna(),
    clean_df["Promotion"].fillna(-1)
)

Promotion,-1.0,0.0,1.0
Discount_Percent,,,
False,48,13953,2606
True,2,113,18


In [12]:
clean_df["Discount_Percent"] = clean_df["Discount_Percent"].fillna(0)

In [13]:
print(
    "Missing Discount_Percent:",
    clean_df["Discount_Percent"].isnull().sum()
)

Missing Discount_Percent: 0


Handle Promotion
- there is no indication that a missing promotion means something different, we'll treat missing values as no promotion.

In [14]:
clean_df["Promotion"] = clean_df["Promotion"].fillna(0)

In [15]:
clean_df["Promotion"] = clean_df["Promotion"].astype(int)

In [16]:
print(clean_df["Promotion"].value_counts())

Promotion
0    14116
1     2624
Name: count, dtype: int64


Handle Lead_Time_Days
- lead time is associated with the product/supplier, use the product-level median.

In [17]:
clean_df["Lead_Time_Days"] = (
    clean_df.groupby("Product_ID")["Lead_Time_Days"]
    .transform(lambda x: x.fillna(x.median()))
)

In [18]:
print(
    "Missing Lead_Time_Days:",
    clean_df["Lead_Time_Days"].isnull().sum()
)

Missing Lead_Time_Days: 0


Handle Current_Stock
- we'll use the product-level median stock.

In [19]:
clean_df["Current_Stock"] = (
    clean_df.groupby("Product_ID")["Current_Stock"]
    .transform(lambda x: x.fillna(x.median()))
)

In [20]:
print(
    "Missing Current_Stock:",
    clean_df["Current_Stock"].isnull().sum()
)

Missing Current_Stock: 0


In [21]:
missing_summary = clean_df.isnull().sum()

print(missing_summary)

Date                0
Product_ID          0
Product_Name        0
Category            0
Brand               0
Current_Stock       0
Units_Sold          0
Unit_Price          0
Discount_Percent    0
Promotion           0
Holiday             0
Lead_Time_Days      0
Safety_Stock        0
Stockout            0
dtype: int64


In [22]:
print("Negative Units Sold:", (clean_df["Units_Sold"] < 0).sum())
print("Negative Stock:", (clean_df["Current_Stock"] < 0).sum())
print("Negative Price:", (clean_df["Unit_Price"] < 0).sum())
print("Invalid Discount:", (clean_df["Discount_Percent"] < 0).sum())
print("Invalid Lead Time:", (clean_df["Lead_Time_Days"] <= 0).sum())
print("Invalid Safety Stock:", (clean_df["Safety_Stock"] < 0).sum())

Negative Units Sold: 0
Negative Stock: 0
Negative Price: 0
Invalid Discount: 0
Invalid Lead Time: 0
Invalid Safety Stock: 0


In [23]:
for col in ["Promotion", "Holiday", "Stockout"]:
    print(f"\n{col}:")
    print(clean_df[col].value_counts())


Promotion:
Promotion
0    14116
1     2624
Name: count, dtype: int64

Holiday:
Holiday
0    16560
1      180
Name: count, dtype: int64

Stockout:
Stockout
0    16484
1      256
Name: count, dtype: int64


In [24]:
output_path = "../data/processed/cleaned_inventory.csv"

clean_df.to_csv(output_path, index=False)

print(f"Cleaned dataset saved to: {output_path}")
print("Final shape:", clean_df.shape)

Cleaned dataset saved to: ../data/processed/cleaned_inventory.csv
Final shape: (16740, 14)


In [25]:
print("========== FINAL DATA QUALITY CHECK ==========")

print("Rows:", clean_df.shape[0])
print("Columns:", clean_df.shape[1])
print("Duplicates:", clean_df.duplicated().sum())
print("Missing values:", clean_df.isnull().sum().sum())

========== FINAL DATA QUALITY CHECK ==========
Rows: 16740
Columns: 14
Duplicates: 0
Missing values: 0
